# Møller vortex packets — analysis workspace

This notebook is intended for day-to-day calculations based on the local `moller_vortex` package.

Use it for parameter scans, quick checks, plots, and comparisons between analytic and numerical expressions. The implementation itself lives in `src/moller_vortex/`; built-in check functions live in `src/moller_vortex/checks.py`; the detailed function guide is in `docs/FUNCTION_GUIDE.md`.


## 1. Imports and local package path

The cell below makes the local `src/` directory importable when the notebook is run from the project root. If the package has been installed with `pip install -e .`, this path setup is harmless.

In [ ]:
import numpy as np
import moller_vortex as mv
from moller_vortex import *


## 2. Numerical accuracy

All adaptive integrations use a `NumericalAccuracy` object. Keep accuracy parameters explicit in scans so that numerical settings are reproducible.

In [ ]:
accuracy = mv.NumericalAccuracy(
    quad_epsabs=1.0e-10,
    quad_epsrel=1.0e-10,
    quad_limit=300,
    root_residual_atol=1.0e-10,
)

accuracy


## 2.1 Built-in consistency checks

Run these when you want to check that the main analytic and numerical pieces still agree. They are ordinary functions, not a separate pytest suite.


In [ ]:
check_results = mv.run_all_checks(
    accuracy=accuracy,
    n_phi=16,
    verbose=True,
)

check_results


## 3. Incoming packets and normalization

`LGPacket` contains only physical packet parameters. Normalization constants are computed explicitly by `normalization_constant`.

In [ ]:
packet1 = mv.LGPacket(
    ell=1,
    sigma_perp=0.18,
    sigma_par=0.35,
    kbar_z=20.0,
)

packet2 = mv.LGPacket(
    ell=-1,
    sigma_perp=0.18,
    sigma_par=0.35,
    kbar_z=-20.0,
)

N1 = mv.normalization_constant(packet1, accuracy=accuracy)
N2 = mv.normalization_constant(packet2, accuracy=accuracy)

print("packet1 =", packet1)
print("packet2 =", packet2)
print("N1 =", N1)
print("N2 =", N2)

## 4. Final momenta, helicities, and impact parameter

All momenta are in MeV. The impact parameter is in MeV$^{-1}$. In the current convention the impact phase is applied only to the second incoming packet.

In [ ]:
k3 = np.array([0.80, 0.10, 19.70])
k4 = np.array([-0.55, -0.08, -19.60])

impact_b = np.array([0.30, 0.00])

lam1 = 0.5
lam2 = 0.5
lam3 = 0.5
lam4 = 0.5

## 5. Closed impulse S-matrix

This is the main fast expression. It uses the analytic transverse integral.

In [ ]:
S_closed = mv.S_impulse_closed_form(
    k3,
    k4,
    packet1,
    packet2,
    lam1=lam1,
    lam2=lam2,
    lam3=lam3,
    lam4=lam4,
    impact_b=impact_b,
    N1=N1,
    N2=N2,
    accuracy=accuracy,
)

S_closed

## 6. Numerical transverse check

This is slower. It replaces the analytic transverse integral by a direct numerical transverse integration, while keeping the same external prefactors. Use it as a check, not as the main production method.

In [ ]:
S_numeric_transverse = mv.S_impulse_numeric_transverse_quad(
    k3,
    k4,
    packet1,
    packet2,
    lam1=lam1,
    lam2=lam2,
    lam3=lam3,
    lam4=lam4,
    impact_b=impact_b,
    N1=N1,
    N2=N2,
    n_phi=32,
    accuracy=accuracy,
)

print("S_closed             =", S_closed)
print("S_numeric_transverse =", S_numeric_transverse)
print("relative error       =", mv.relative_error(S_numeric_transverse, S_closed))

## 7. Scan over discrete OAM charge

Example scan over $\ell_1$ with all other parameters fixed.

In [ ]:
ell_values = np.arange(-4, 5)
scan_ell = []

for ell in ell_values:
    p1 = mv.LGPacket(
        ell=int(ell),
        sigma_perp=packet1.sigma_perp,
        sigma_par=packet1.sigma_par,
        kbar_z=packet1.kbar_z,
    )
    Np1 = mv.normalization_constant(p1, accuracy=accuracy)
    S_ell = mv.S_impulse_closed_form(
        k3,
        k4,
        p1,
        packet2,
        lam1=lam1,
        lam2=lam2,
        lam3=lam3,
        lam4=lam4,
        impact_b=impact_b,
        N1=Np1,
        N2=N2,
        accuracy=accuracy,
    )
    scan_ell.append((ell, S_ell, abs(S_ell)))

for ell, S_ell, abs_S in scan_ell:
    print(f"ell1={ell:2d}  S={S_ell}  |S|={abs_S:.6e}")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot([row[0] for row in scan_ell], [row[2] for row in scan_ell], marker="o")
plt.xlabel(r"$\ell_1$")
plt.ylabel(r"$|S_{fi}|$")
plt.title("Scan over incoming OAM charge")
plt.grid(True)
plt.show()

## 8. Scan over continuous width parameter

Example scan over $\sigma_\perp$ for the first packet. The condition $\sigma_\perp \le \sigma_\parallel$ must be respected.

In [ ]:
sigma_values = np.linspace(0.10, packet1.sigma_par, 12)
scan_sigma = []

for sigma_perp in sigma_values:
    p1 = mv.LGPacket(
        ell=packet1.ell,
        sigma_perp=float(sigma_perp),
        sigma_par=packet1.sigma_par,
        kbar_z=packet1.kbar_z,
    )
    Np1 = mv.normalization_constant(p1, accuracy=accuracy)
    S_sigma = mv.S_impulse_closed_form(
        k3,
        k4,
        p1,
        packet2,
        lam1=lam1,
        lam2=lam2,
        lam3=lam3,
        lam4=lam4,
        impact_b=impact_b,
        N1=Np1,
        N2=N2,
        accuracy=accuracy,
    )
    scan_sigma.append((sigma_perp, S_sigma, abs(S_sigma)))

plt.figure(figsize=(6, 4))
plt.plot([row[0] for row in scan_sigma], [row[2] for row in scan_sigma], marker="o")
plt.xlabel(r"$\sigma_{1\perp}$ [MeV]")
plt.ylabel(r"$|S_{fi}|$")
plt.title("Scan over transverse width")
plt.grid(True)
plt.show()

## 9. Notes for further work

For production scans, avoid recomputing normalization constants unnecessarily. Cache them in dictionaries keyed by parameter tuples, for example `(ell, sigma_perp, sigma_par, kbar_z)`.

Keep physical calculations in functions and use the notebook cells for parameter choices, plots, and diagnostics.